In [2]:
# ============================================================
# Load saved NPHA CSV, convert numeric codes to categorical
# labels, and save a new AVISTA-ready categorical dataset
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Load existing saved dataset
# ------------------------------------------------------------

input_path = Path("data/uci_npha_avista.csv")

df = pd.read_csv(input_path)

print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())


# ------------------------------------------------------------
# 2. Drop Age if still present
# It was constant in this dataset
# ------------------------------------------------------------

if "Age" in df.columns:
    print("\nAge unique values:", df["Age"].unique())

    if df["Age"].nunique(dropna=False) == 1:
        df = df.drop(columns=["Age"])
        print("Dropped constant feature: Age")


# ------------------------------------------------------------
# 3. Categorical mappings
# ------------------------------------------------------------

mappings = {

    "Physical_Health": {
        -1: "Refused",
        1: "Excellent",
        2: "Very_Good",
        3: "Good",
        4: "Fair",
        5: "Poor"
    },

    "Mental_Health": {
        -1: "Refused",
        1: "Excellent",
        2: "Very_Good",
        3: "Good",
        4: "Fair",
        5: "Poor"
    },

    "Dental_Health": {
        -1: "Refused",
        1: "Excellent",
        2: "Very_Good",
        3: "Good",
        4: "Fair",
        5: "Poor"
    },

    "Employment": {
        -1: "Refused",
        1: "Working_Full_Time",
        2: "Working_Part_Time",
        3: "Retired",
        4: "Not_Working"
    },

    "Stress_Keeps_Patient_from_Sleeping": {
        0: "No",
        1: "Yes"
    },

    "Medication_Keeps_Patient_from_Sleeping": {
        0: "No",
        1: "Yes"
    },

    "Pain_Keeps_Patient_from_Sleeping": {
        0: "No",
        1: "Yes"
    },

    "Bathroom_Needs_Keeps_Patient_from_Sleeping": {
        0: "No",
        1: "Yes"
    },

    "Uknown_Keeps_Patient_from_Sleeping": {
        0: "No",
        1: "Yes"
    },

    "Prescription_Sleep_Medication": {
        -1: "Refused",
        1: "Use_Regularly",
        2: "Use_Occasionally",
        3: "Do_Not_Use"
    },

    "Race": {
        -2: "Not_Asked",
        -1: "Refused",
        1: "White_Non_Hispanic",
        2: "Black_Non_Hispanic",
        3: "Other_Non_Hispanic",
        4: "Hispanic",
        5: "Two_or_More_Races"
    },

    "Gender": {
        -2: "Not_Asked",
        -1: "Refused",
        1: "Male",
        2: "Female"
    },

    "Number_of_Doctors_Visited": {
        1: "Doctors_0_1",
        2: "Doctors_2_3",
        3: "Doctors_4_Plus"
    }
}


# ------------------------------------------------------------
# 4. Apply mappings
# ------------------------------------------------------------

for col, mapping in mappings.items():

    if col in df.columns:

        original = df[col].copy()

        converted = original.map(mapping)

        # Detect unexpected/unmapped values
        unmapped = original[
            converted.isna() & original.notna()
        ].unique()

        if len(unmapped) > 0:
            print(
                f"WARNING - {col}: "
                f"unmapped values {sorted(unmapped.tolist())}"
            )

        df[col] = converted


# ------------------------------------------------------------
# 5. Trouble_Sleeping
#
# Actual dataset contains values such as 2 and 3, while the
# UCI description appears inconsistent.
# Preserve codes as explicit categorical labels.
# ------------------------------------------------------------

if "Trouble_Sleeping" in df.columns:

    df["Trouble_Sleeping"] = (
        "Trouble_Sleeping_"
        + df["Trouble_Sleeping"].astype(str)
    )


# ------------------------------------------------------------
# 6. Force all remaining variables to categorical/text
# ------------------------------------------------------------

for col in df.columns:
    df[col] = df[col].astype("string")


# ------------------------------------------------------------
# 7. Verify
# ------------------------------------------------------------

print("\n=== CONVERTED DATASET ===")
print(f"Rows     : {df.shape[0]}")
print(f"Features : {df.shape[1] - 1}")
print(f"Target   : Number_of_Doctors_Visited")

print("\nTarget distribution:")
print(df["Number_of_Doctors_Visited"].value_counts())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

display(df.head())


# ------------------------------------------------------------
# 8. Save categorical AVISTA dataset
# ------------------------------------------------------------

output_path = Path(
    "data/uci_npha_avista_categorical.csv"
)

df.to_csv(
    output_path,
    index=False
)

print(f"\nSaved to: {output_path}")

# Reload verification
check = pd.read_csv(output_path)

print("\n=== SAVED FILE CHECK ===")
print("Shape:", check.shape)
print(
    "Target values:",
    check["Number_of_Doctors_Visited"].unique()
)

Dataset loaded successfully.
Rows    : 714
Columns : 14


,Physical_Health,Mental_Health,Dental_Health,Employment,Stress_Keeps_Patient_from_Sleeping,Medication_Keeps_Patient_from_Sleeping,Pain_Keeps_Patient_from_Sleeping,Bathroom_Needs_Keeps_Patient_from_Sleeping,Uknown_Keeps_Patient_from_Sleeping,Trouble_Sleeping,Prescription_Sleep_Medication,Race,Gender,Number_of_Doctors_Visited
0,4,3,3,3,0,0,0,0,1,2,3,1,2,3
1,4,2,3,3,1,0,0,1,0,3,3,1,1,2
2,3,2,3,3,0,0,0,0,1,3,3,4,1,3
3,3,2,3,3,0,0,0,1,0,3,3,4,2,1
4,3,3,3,3,1,0,0,0,0,2,3,1,2,3


WARNING - Dental_Health: unmapped values [6]

=== CONVERTED DATASET ===
Rows     : 714
Features : 13
Target   : Number_of_Doctors_Visited

Target distribution:
Number_of_Doctors_Visited
Doctors_2_3       372
Doctors_4_Plus    211
Doctors_0_1       131
Name: count, dtype: Int64

Data types:


,dtype
Physical_Health,string[python]
Mental_Health,string[python]
Dental_Health,string[python]
Employment,string[python]
Stress_Keeps_Patient_from_Sleeping,string[python]
Medication_Keeps_Patient_from_Sleeping,string[python]
Pain_Keeps_Patient_from_Sleeping,string[python]
Bathroom_Needs_Keeps_Patient_from_Sleeping,string[python]
Uknown_Keeps_Patient_from_Sleeping,string[python]
Trouble_Sleeping,string[python]


,Physical_Health,Mental_Health,Dental_Health,Employment,Stress_Keeps_Patient_from_Sleeping,Medication_Keeps_Patient_from_Sleeping,Pain_Keeps_Patient_from_Sleeping,Bathroom_Needs_Keeps_Patient_from_Sleeping,Uknown_Keeps_Patient_from_Sleeping,Trouble_Sleeping,Prescription_Sleep_Medication,Race,Gender,Number_of_Doctors_Visited
0,Fair,Good,Good,Retired,No,No,No,No,Yes,Trouble_Sleeping_2,Do_Not_Use,White_Non_Hispanic,Female,Doctors_4_Plus
1,Fair,Very_Good,Good,Retired,Yes,No,No,Yes,No,Trouble_Sleeping_3,Do_Not_Use,White_Non_Hispanic,Male,Doctors_2_3
2,Good,Very_Good,Good,Retired,No,No,No,No,Yes,Trouble_Sleeping_3,Do_Not_Use,Hispanic,Male,Doctors_4_Plus
3,Good,Very_Good,Good,Retired,No,No,No,Yes,No,Trouble_Sleeping_3,Do_Not_Use,Hispanic,Female,Doctors_0_1
4,Good,Good,Good,Retired,Yes,No,No,No,No,Trouble_Sleeping_2,Do_Not_Use,White_Non_Hispanic,Female,Doctors_4_Plus



Saved to: data\uci_npha_avista_categorical.csv

=== SAVED FILE CHECK ===
Shape: (714, 14)
Target values: ['Doctors_4_Plus' 'Doctors_2_3' 'Doctors_0_1']
